# Supermarket Sales Data Cleaning and Preparation

**Task 1 — Data Cleaning & Preparation**

This notebook cleans the supplied `SuperMarket Analysis.csv` dataset. It checks data quality, normalizes date/time formats, converts numeric fields, trims text whitespace, checks duplicates, and validates key sales calculations.

**Dataset:** 1,000 rows and 17 columns in the supplied file. The source file is included in `../data/SuperMarket_Analysis_Raw.csv`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# In Google Colab, upload SuperMarket_Analysis_Raw.csv or change this path.
input_path = Path("../data/SuperMarket_Analysis_Raw.csv")
df = pd.read_csv(input_path)

print("Rows and columns:", df.shape)
display(df.head())
print("\\nMissing values by column:")
display(df.isna().sum())
print("\\nExact duplicate rows:", df.duplicated().sum())
print("\\nOriginal data types:")
display(df.dtypes)

## Cleaning decisions

- Trim leading/trailing whitespace in text fields.
- Parse the mixed date formats into a consistent date format.
- Convert time values to a consistent 24-hour format.
- Convert numeric fields to numeric types.
- Remove exact duplicate rows only. Similar-looking records should not be removed without evidence that they are duplicates.
- Check sales, tax, and gross-income calculations. Do not change valid business values merely because they look unusual.

In [ ]:
clean = df.copy()

# Trim whitespace in all text columns.
text_cols = clean.select_dtypes(include="object").columns.tolist()
for col in text_cols:
    clean[col] = clean[col].astype("string").str.strip()

# Parse mixed date formats; invalid values become NaT and are counted below.
clean["Date"] = pd.to_datetime(clean["Date"], format="mixed", errors="coerce", dayfirst=False)

# Convert time values from 12-hour text to 24-hour HH:MM:SS.
clean["Time"] = pd.to_datetime(
    clean["Time"], format="%I:%M:%S %p", errors="coerce"
).dt.strftime("%H:%M:%S")

numeric_cols = [
    "Unit price", "Quantity", "Tax 5%", "Sales", "cogs",
    "gross margin percentage", "gross income", "Rating"
]
for col in numeric_cols:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

# Remove exact duplicates only.
duplicates_removed = int(clean.duplicated().sum())
clean = clean.drop_duplicates().copy()

print("Exact duplicate rows removed:", duplicates_removed)
print("Invalid/unparsed dates:", int(clean["Date"].isna().sum()))
print("Invalid/unparsed times:", int(clean["Time"].isna().sum()))
print("Missing cells after type conversion:")
display(clean.isna().sum())

In [ ]:
# Validate core calculations using a small tolerance for floating-point rounding.
sales_mismatch = ~np.isclose(
    clean["Sales"], clean["Unit price"] * clean["Quantity"] * 1.05,
    rtol=1e-5, atol=0.02
)
tax_mismatch = ~np.isclose(
    clean["Tax 5%"], clean["cogs"] * 0.05, rtol=1e-5, atol=0.02
)
income_mismatch = ~np.isclose(
    clean["gross income"], clean["Tax 5%"], rtol=1e-5, atol=0.02
)

print("Sales formula mismatches:", int(sales_mismatch.sum()))
print("Tax formula mismatches:", int(tax_mismatch.sum()))
print("Gross income vs tax mismatches:", int(income_mismatch.sum()))

# Show category values to inspect consistency; this dataset has only a small set of labels.
for col in ["Branch", "City", "Customer type", "Gender", "Product line", "Payment"]:
    print(f"{col}: {sorted(clean[col].dropna().unique().tolist())}")

In [ ]:
# Export the cleaned data with a consistent date representation.
clean["Date"] = clean["Date"].dt.strftime("%Y-%m-%d")
output_path = Path("../data/SuperMarket_Analysis_Cleaned.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(output_path, index=False)

print("Saved cleaned dataset to:", output_path)
print("Final shape:", clean.shape)
print("Final missing cells:", int(clean.isna().sum().sum()))
display(clean.head())

## Results for the supplied file

- Original size: **1,000 rows × 17 columns**
- Missing values in the original file: **0**
- Exact duplicate rows in the original file: **0**
- Dates parsed successfully: **all 1,000**
- Times parsed successfully: **all 1,000**
- Sales/tax/gross-income formula mismatches: **0** for each check
- Date format in the output: `YYYY-MM-DD`
- Time format in the output: `HH:MM:SS` (24-hour clock)

No rows were removed because there were no exact duplicates. The dataset's date strings used mixed formats, so dates were normalized. No categorical labels were renamed because the inspected category fields already had consistent distinct values. Always review any flagged values in context before changing them.